# Lab 02: LSTM and GRU Comparison

**Module 02** | Read `notes/02-lstm-gru.pdf` before running this notebook.

Train identical LSTM and GRU classifiers on the same data and compare final loss, wall-clock time, and parameter count.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **LSTM** | Long Short-Term Memory: an RNN variant with gates that control what to remember and forget. |
| **GRU** | Gated Recurrent Unit: a simpler gated RNN with fewer parameters than LSTM, often similarly effective. |
| **gate** | A learned switch (0 to 1) that decides how much old memory to keep vs. how much new input to add. |
| **vanishing gradient** | A problem where gradients shrink to near zero over many timesteps, slowing learning of long-range patterns. |
| **parameter count** | The total number of trainable weights in a model. More parameters usually means more capacity but slower training. |

Refer back to this table whenever a term appears in code or output.


## What problem are we solving?

Plain RNNs forget information from early timesteps because gradients shrink as they flow backward through many steps. **LSTM** and **GRU** cells add **gates** that control memory flow, which usually helps with longer dependencies.

## What you will learn

- Why **vanishing gradients** limit vanilla RNNs
- How LSTM and GRU differ in structure and **parameter count**
- How to run a fair comparison on the **same data and task** as Lab 01
- How to read side-by-side predictions from both architectures

We keep everything identical to Lab 01 except the recurrent cell, so any difference in results comes from the architecture, not the dataset.


### Step 1: Rebuild the Lab 01 data pipeline

**What this code does:** Recreates the exact same corpus, vocabulary, and sliding-window dataset from Lab 01.

**Why we do it:** Fair comparison requires identical data. If we changed the corpus, we could not tell whether a metric gap came from the cell type or from different examples.


**What to look for in the output**

Example count in the thousands and vocabulary size matching Lab 01 (around 20 characters).


In [ ]:
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("..").resolve()
SEQ_LEN = 32
BATCH_SIZE = 64
HIDDEN_SIZE = 128
EPOCHS = 5
LR = 1e-2

# Same corpus and preprocessing as Lab 01
CORPUS = "sequence modeling practice text " * 50
chars = sorted(set(CORPUS))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

indices = torch.tensor([char_to_idx[c] for c in CORPUS], dtype=torch.long)
inputs, targets = [], []
for start in range(len(indices) - SEQ_LEN):
    inputs.append(indices[start : start + SEQ_LEN])
    targets.append(indices[start + SEQ_LEN])
inputs = torch.stack(inputs)
targets = torch.stack(targets)

dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Examples: {len(dataset):,} | Vocab: {vocab_size}")


**Concept: Gates and vanishing gradients**

A **gate** is a learned value between 0 and 1 that controls information flow. LSTM has separate forget, input, and output gates plus a **cell state** (long-term memory). GRU merges some of that into update and reset gates with fewer weights.

**Vanishing gradients** happen when repeated multiplication during backprop makes gradients shrink toward zero. Gates help by creating paths where gradients can flow more reliably across timesteps.


### Step 2: Define LSTM and GRU classifiers

**What this code does:** Defines a shared model class that swaps `nn.LSTM` or `nn.GRU` as the recurrent core, plus a reusable training function.

**Why we do it:** Identical outer structure (embedding + recurrent core + linear head) isolates the effect of the cell type on loss, speed, and parameter count.


**What to look for in the output**

No output yet from this cell alone; it defines classes and a training helper for the next step.


In [ ]:
class CharRecurrentClassifier(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, cell: str = "lstm"):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        if cell == "lstm":
            # LSTM maintains separate hidden state and cell state
            self.rnn = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        elif cell == "gru":
            # GRU uses a single hidden state with gating
            self.rnn = nn.GRU(hidden_size, hidden_size, batch_first=True)
        else:
            raise ValueError(f"Unknown cell: {cell}")
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.cell = cell

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embed(x)
        output, _ = self.rnn(embedded)
        return self.fc(output[:, -1, :])


def decode_indices(idxs: torch.Tensor) -> str:
    return "".join(idx_to_char[i.item()] for i in idxs)


def train_model(model: nn.Module) -> tuple[float, list[float]]:
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_per_epoch = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch_x.size(0)
        avg_loss = epoch_loss / len(dataset)
        loss_per_epoch.append(avg_loss)
        print(f"  Epoch {epoch:02d}/{EPOCHS}, loss {avg_loss:.4f}")
    return loss_per_epoch[-1], loss_per_epoch


### Step 3: Train LSTM and GRU with per-epoch logging

**What this code does:** Trains both models with the same random seed, records final loss, wall-clock time, and parameter count for each.

**Why we do it:** Numbers tell us which cell converges faster or fits better on this task. Fixing the seed ensures both models start from the same random weights.


**What to look for in the output**

Two training runs with per-epoch loss lines, then a comparison table. LSTM typically has slightly more parameters than GRU. Both final losses should be low on this easy corpus.


In [ ]:
results = {}
trained_models = {}

for name, cell in [("LSTM", "lstm"), ("GRU", "gru")]:
    print(f"\nTraining {name}...")
    model = CharRecurrentClassifier(vocab_size, HIDDEN_SIZE, cell=cell)
    params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {params:,}")

    # Same seed so weight initialization does not bias the comparison
    torch.manual_seed(42)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(42)

    start = time.perf_counter()
    final_loss, loss_history = train_model(model)
    elapsed = time.perf_counter() - start
    results[name] = {"final_loss": final_loss, "elapsed": elapsed, "params": params, "loss_history": loss_history}
    trained_models[name] = model

print("\nFinal comparison")
print(f"{'Model':<8} {'Final loss':>12} {'Time (s)':>10} {'Params':>10}")
print("-" * 44)
for name, r in results.items():
    print(f"{name:<8} {r['final_loss']:>12.4f} {r['elapsed']:>10.2f} {r['params']:>10,}")


### Step 4: Side-by-side predictions on the same examples

**What this code does:** Runs both trained models on the same five input windows and marks each prediction as correct or wrong.

**Why we do it:** Loss and timing summarize overall behavior; per-example predictions show whether both models agree or disagree on specific patterns.


**What to look for in the output**

A table with input snippets, targets, and LSTM/GRU predictions. On this small corpus both models should get most examples right after training.


In [ ]:
@torch.no_grad()
def predict_char(model: nn.Module, input_seq: torch.Tensor) -> tuple[str, int]:
    model.eval()
    logits = model(input_seq.unsqueeze(0).to(device))
    pred_idx = logits.argmax(dim=-1).item()
    return idx_to_char[pred_idx], pred_idx


def sample_eval_indices(n: int, k: int = 5) -> list[int]:
    """Pick k spread-out row indices in [0, n-1] without going out of bounds."""
    if n <= 0:
        return [0]
    if n <= k:
        return list(range(n))
    return sorted({0, n // 4, n // 2, (3 * n) // 4, n - 1})

eval_indices = sample_eval_indices(len(inputs))
print("Predictions on 5 shared examples:")
print("-" * 72)
print(f"{'#':<4} {'Input (last 12)':<16} {'Target':<8} {'LSTM':<14} {'GRU':<14}")
print("-" * 72)

for row, i in enumerate(eval_indices):
    window = decode_indices(inputs[i])
    short = window[-12:]
    target_ch = idx_to_char[targets[i].item()]
    lstm_ch, _ = predict_char(trained_models["LSTM"], inputs[i])
    gru_ch, _ = predict_char(trained_models["GRU"], inputs[i])
    lstm_mark = "OK" if lstm_ch == target_ch else "WRONG"
    gru_mark = "OK" if gru_ch == target_ch else "WRONG"
    print(f"{row:<4} {short!r:<16} {target_ch!r:<8} {lstm_ch!r} ({lstm_mark:<5}) {gru_ch!r} ({gru_mark})")


### Summary

LSTM and GRU solve the same next-character task with nearly identical code. Compare final loss, training time, parameter count, and per-example predictions. On longer sequences and larger vocabularies, gated cells often outperform plain RNNs more clearly.


In [ ]:
print("=" * 60)
print("Lab 02 summary")
print("=" * 60)
for name, r in results.items():
    print(f"  {name}: loss={r['final_loss']:.4f}, time={r['elapsed']:.2f}s, params={r['params']:,}")
winner = min(results, key=lambda k: results[k]["final_loss"])
print(f"  Lowest final loss: {winner}")
print("=" * 60)
